# This file finds unique foods


In [1]:
import os
import pandas as pd
import hashlib
import math

In [2]:
def hash_file(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

In [3]:
base_folder = "../data/grand_scraper_folder/scrandle_data"

In [4]:
unique_cases = []
seen_metadata = {}


# Helper: normalize values BEFORE using as key
def normalize(row):
    title = str(row["title"]).strip()

    subtitle = row["subtitle"]
    if pd.isna(subtitle):
        subtitle = ""
    else:
        subtitle = str(subtitle).strip()

    price = row["price"]
    if pd.isna(price):
        price = None
    else:
        price = float(price)

    rating = row["rating"]
    if pd.isna(rating):
        rating = None
    else:
        rating = float(rating)

    return title, subtitle, price, rating


for date in sorted(os.listdir(base_folder)):
    
    date_path = os.path.join(base_folder, date)
    
    if not os.path.isdir(date_path):
        continue

    print(f"Processing {date}")
    
    meta_path = os.path.join(date_path, "meta.csv")
    
    if not os.path.exists(meta_path):
        print(f"⚠️ Missing meta.csv for {date}")
        continue

    df = pd.read_csv(meta_path)

    for _, row in df.iterrows():

        # ✅ NORMALIZED metadata key
        meta_key = normalize(row)

        # ---- Identify image file
        image_name = row["image_name"]
        image_path = os.path.join(date_path, image_name)

        if not os.path.exists(image_path):
            print(f"⚠️ Missing image: {image_path}")
            continue

        # ---- Compute hash
        img_hash = hash_file(image_path)

        # ---- Occurrence string
        occurrence = f"{date}:{row['match_id']}_{row['side']}"

        
        # =========================
        # CASE 1: NEW METADATA
        # =========================
        if meta_key not in seen_metadata:
            
            seen_metadata[meta_key] = [{
                "hash": img_hash,
                "index": len(unique_cases)
            }]

            unique_cases.append({
                "title": meta_key[0],
                "subtitle": meta_key[1],
                "price": meta_key[2],
                "rating": meta_key[3],
                "image_hash": img_hash,
                "occurrences": [occurrence]
            })


        # =========================
        # CASE 2: METADATA EXISTS
        # =========================
        else:
            found_match = False

            for entry in seen_metadata[meta_key]:
                
                if entry["hash"] == img_hash:
                    idx = entry["index"]
                    unique_cases[idx]["occurrences"].append(occurrence)
                    found_match = True
                    break

            # SAME metadata BUT different image
            if not found_match:
                seen_metadata[meta_key].append({
                    "hash": img_hash,
                    "index": len(unique_cases)
                })

                unique_cases.append({
                    "title": meta_key[0],
                    "subtitle": meta_key[1],
                    "price": meta_key[2],
                    "rating": meta_key[3],
                    "image_hash": img_hash,
                    "occurrences": [occurrence]
                })

Processing 2025-04-20
Processing 2025-04-21
Processing 2025-04-22
Processing 2025-04-23
Processing 2025-04-24
Processing 2025-04-25
Processing 2025-04-26
Processing 2025-04-27
Processing 2025-04-28
Processing 2025-04-29
Processing 2025-04-30
Processing 2025-05-01
Processing 2025-05-02
Processing 2025-05-03
Processing 2025-05-04
Processing 2025-05-05
Processing 2025-05-06
Processing 2025-05-07
Processing 2025-05-08
Processing 2025-05-09
Processing 2025-05-10
Processing 2025-05-11
Processing 2025-05-12
Processing 2025-05-13
Processing 2025-05-14
Processing 2025-05-15
Processing 2025-05-16
Processing 2025-05-17
Processing 2025-05-18
Processing 2025-05-19
Processing 2025-05-20
Processing 2025-05-21
Processing 2025-05-22
Processing 2025-05-23
Processing 2025-05-24
Processing 2025-05-25
Processing 2025-05-26
Processing 2025-05-27
Processing 2025-05-28
Processing 2025-05-29
Processing 2025-05-30
Processing 2025-05-31
Processing 2025-06-01
Processing 2025-06-02
Processing 2025-06-03
Processing

In [5]:
df_unique = pd.DataFrame(unique_cases)

# Make occurrences readable
df_unique["occurrences"] = df_unique["occurrences"].apply(lambda x: " | ".join(x))

df_unique

,title,subtitle,price,rating,image_hash,occurrences
0,Hot dog in a waffle,,2.60,25.80,16e31e839190d590eaf12295d90e8e21,2025-04-20:0_left | 2025-04-29:8_right | 2025-...
1,Chicharrones,Fried pork belly with guac and chimichurri,9.50,90.60,e2bc591910ad1e91cfe9a3f55c90be74,2025-04-20:0_right
2,Sánguche de potito,Cows rectum sandwich,2.80,32.50,0cc3106abed465439d5dfe7fb0f7815e,2025-04-20:1_left | 2025-05-18:2_right | 2025-...
3,Turkey schnitzel,,3.40,81.00,e9db7022b5a276a790d0b28f7e8fd77b,2025-04-20:1_right | 2025-08-18:3_right | 2025...
4,Katsu chicken curry,,5.50,91.20,972c80a83ed064937093e776361e83b6,2025-04-20:2_left | 2025-06-01:7_right | 2025-...
...,...,...,...,...,...,...
4583,Chicken club sandwich,,5.20,14.94,ef1e9cd7290019321e0522eb0895da52,2026-05-19:7_right
4584,Polish dog,,3.40,28.71,246124c2a95bad3e207c7c2c575fd1a3,2026-05-19:8_left
4585,American hot dog,"Pickles, fried onions, mustard, ketchup, 0,5l ...",5.50,55.95,0c18da3fa2b4aa0329f848b83325e004,2026-05-19:8_right
4586,Strawberry daiquiri / pina colada mix,,16.45,52.41,1b7bad06896cd389479b0659f810aff7,2026-05-19:9_left


In [6]:
output_path = os.path.join("..", "data", "grand_scraper_folder", "unique_scrandle_cases_2.csv")
df_unique.to_csv(output_path, index=False)
